[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1yece5-xOZScPEJ3_oUipU76nmMgXpVpQ)

# Overview

We'll try probably one of the oldest problem statements in NLP – **Spam or Ham**

## Data
https://www.dropbox.com/s/6ehj0gdi7avbiqj/SMSSpamCollection.tsv?raw=1

### Example
**Test Message 1 (Spam)**:

"Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"


**Test Message 2 (Ham)**:

"I've been searching for the right words to thank you for this breather"


In [3]:
!wget https://www.dropbox.com/s/6ehj0gdi7avbiqj/SMSSpamCollection.tsv?raw=1 -nc -O SMSSpamCollection.tsv

--2018-11-14 23:29:49--  https://www.dropbox.com/s/6ehj0gdi7avbiqj/SMSSpamCollection.tsv?raw=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.9.1, 2620:100:601f:1::a27d:901
Connecting to www.dropbox.com (www.dropbox.com)|162.125.9.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /s/raw/6ehj0gdi7avbiqj/SMSSpamCollection.tsv [following]
--2018-11-14 23:29:49--  https://www.dropbox.com/s/raw/6ehj0gdi7avbiqj/SMSSpamCollection.tsv
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://ucde874f570be13be07aa2d1f52c.dl.dropboxusercontent.com/cd/0/inline/AVkHHDNzH3xr6aLgw5eqd3ATN6PeWUseLaP-n7AmZadIYDv7IF98-rLe6wt6NlW7yhbMFqCO0U4evY4GHlpPYPGAHIwTSx3a1LekTb7vPWZ6WpK7L5HOGU1SmdtTDDkoZKIUdcV0Jcf5diITW6zZpb-HW5k0DRHY6sfsrlWFBdhDnpoHjibtivTi82EQ9CI8lEg/file [following]
--2018-11-14 23:29:49--  https://ucde874f570be13be07aa2d1f52c.dl.dropboxusercontent.com/cd/0/inline/AVkHHDNzH3xr6aLgw5

# Setup Spark

https://mikestaszel.com/2018/03/07/apache-spark-on-google-colaboratory/

In [2]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://apache.osuosl.org/spark/spark-2.3.1/spark-2.3.1-bin-hadoop2.7.tgz
!tar xf spark-2.3.1-bin-hadoop2.7.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-2.3.1-bin-hadoop2.7"

import findspark
findspark.init()
from pyspark.context import SparkContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = SparkContext.getOrCreate()

sc.getConf().getAll()

[('spark.driver.port', '39273'),
 ('spark.rdd.compress', 'True'),
 ('spark.app.id', 'local-1542238186390'),
 ('spark.serializer.objectStreamReset', '100'),
 ('spark.master', 'local[*]'),
 ('spark.executor.id', 'driver'),
 ('spark.submit.deployMode', 'client'),
 ('spark.driver.host', '3beeded14a0e'),
 ('spark.ui.showConsoleProgress', 'true'),
 ('spark.app.name', 'pyspark-shell')]

In [4]:
samples = sc.textFile("SMSSpamCollection.tsv")
samples.take(5)

["ham\tI've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.",
 "spam\tFree entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
 "ham\tNah I don't think he goes to usf, he lives around here though",
 'ham\tEven my brother is not like to speak with me. They treat me like aids patent.',
 'ham\tI HAVE A DATE ON SUNDAY WITH WILL!!']

# Model 1 – Using HashingTF

In [5]:
corpus = sc.parallelize(samples.collect())
corpus.take(5)

["ham\tI've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.",
 "spam\tFree entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
 "ham\tNah I don't think he goes to usf, he lives around here though",
 'ham\tEven my brother is not like to speak with me. They treat me like aids patent.',
 'ham\tI HAVE A DATE ON SUNDAY WITH WILL!!']

In [0]:
texts = corpus.map(lambda sample: sample.split("\t")[1].split(" "))

In [7]:
from pyspark.mllib.feature import HashingTF

hashingTF = HashingTF()
tf = hashingTF.transform(texts)
tf.take(5)

[SparseVector(1048576, {1475: 1.0, 70882: 1.0, 151357: 2.0, 154253: 1.0, 163495: 1.0, 173174: 1.0, 231791: 1.0, 235395: 1.0, 238153: 1.0, 241476: 1.0, 250929: 1.0, 270412: 1.0, 276491: 3.0, 463522: 1.0, 479025: 1.0, 486014: 1.0, 488866: 1.0, 494808: 1.0, 550685: 1.0, 578619: 2.0, 622323: 1.0, 648331: 1.0, 702216: 1.0, 706364: 1.0, 724221: 1.0, 789438: 1.0, 837499: 1.0, 910746: 1.0, 935701: 1.0, 990085: 1.0, 1000347: 1.0, 1016101: 1.0, 1031802: 1.0}),
 SparseVector(1048576, {27527: 1.0, 28088: 1.0, 30792: 1.0, 99239: 1.0, 154253: 3.0, 184390: 1.0, 217765: 2.0, 238153: 1.0, 290809: 1.0, 309651: 1.0, 349041: 1.0, 376789: 1.0, 424739: 1.0, 471671: 1.0, 539388: 1.0, 625236: 2.0, 629176: 1.0, 687011: 1.0, 738329: 1.0, 792573: 1.0, 825809: 1.0, 997716: 1.0, 1017170: 1.0, 1045692: 1.0}),
 SparseVector(1048576, {1475: 1.0, 40306: 1.0, 58124: 1.0, 125951: 1.0, 154253: 1.0, 268040: 2.0, 546285: 1.0, 768566: 1.0, 853004: 1.0, 889687: 1.0, 908132: 1.0, 937459: 1.0}),
 SparseVector(1048576, {3121: 1

In [0]:
from pyspark.mllib.regression import LabeledPoint

def labeled_point_per_sample(sample_and_vector):
  sample = sample_and_vector[0]
  feature_vector = sample_and_vector[1]
  label = 1 if sample.split("\t")[0] == 'spam' else 0
  return LabeledPoint(label, feature_vector)

In [9]:
labeled_points_rdd = corpus.zip(tf).map(labeled_point_per_sample)
print("size of data:", len(labeled_points_rdd.collect()))
labeled_points_rdd.take(5)

size of data: 5570


[LabeledPoint(0.0, (1048576,[1475,70882,151357,154253,163495,173174,231791,235395,238153,241476,250929,270412,276491,463522,479025,486014,488866,494808,550685,578619,622323,648331,702216,706364,724221,789438,837499,910746,935701,990085,1000347,1016101,1031802],[1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])),
 LabeledPoint(1.0, (1048576,[27527,28088,30792,99239,154253,184390,217765,238153,290809,309651,349041,376789,424739,471671,539388,625236,629176,687011,738329,792573,825809,997716,1017170,1045692],[1.0,1.0,1.0,1.0,3.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])),
 LabeledPoint(0.0, (1048576,[1475,40306,58124,125951,154253,268040,546285,768566,853004,889687,908132,937459],[1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0])),
 LabeledPoint(0.0, (1048576,[3121,154253,260432,342716,345663,347874,348943,415486,617454,648331,682167,698511,800922,847465,925264],[1.0,1.0,

In [0]:
from pyspark.mllib.classification import LogisticRegressionWithSGD

labeled_points_rdd.cache()
spam_model = LogisticRegressionWithSGD.train(labeled_points_rdd)

### Evaluate

In [11]:
def feature_vector_from_text(text):
  words = text.split(" ")
  tf = hashingTF.transform(words)
  return tf

test_candidates = [feature_vector_from_text("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"),
                   feature_vector_from_text("I've been searching for the right words to thank you for this breather"),
                   feature_vector_from_text("WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only."),
                   feature_vector_from_text("URGENT, IMPORTANT INFORMATION FOR O2 USER. TODAY IS YOUR LUCKY DAY! 2 FIND OUT WHY LOG ONTO HTTP://WWW.URAWINNER.COM THERE IS A FANTASTIC SURPRISE AWAITING FOR YOU"),
                   feature_vector_from_text("THING R GOOD THANX GOT EXAMS IN MARCH IVE DONE NO REVISION? IS FRAN STILL WITH BOYF? IVE GOTTA INTERVIW 4 EXETER BIT WORRIED!x")]
test_data = sc.parallelize(test_candidates)
print(spam_model.predict(test_data).collect())

[0, 0, 0, 0, 0]


**Observation:**
Not really working – 1st, 3rd and 4th test candidates are clearly spam


# Model 2 – Using CountVectorizer with tf-idf

In [12]:
corpus = sc.parallelize(samples.collect())
corpus.take(5)

["ham\tI've been searching for the right words to thank you for this breather. I promise i wont take your help for granted and will fulfil my promise. You have been wonderful and a blessing at all times.",
 "spam\tFree entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's",
 "ham\tNah I don't think he goes to usf, he lives around here though",
 'ham\tEven my brother is not like to speak with me. They treat me like aids patent.',
 'ham\tI HAVE A DATE ON SUNDAY WITH WILL!!']

In [0]:
feature_corpus = corpus.map(lambda each: each.split("\t")[1].split(" "))
feature_corpus_with_index = feature_corpus.zipWithIndex()
feature_corpus_with_index.take(5)

In [14]:
df = feature_corpus_with_index.toDF(["tokens", "id"])
df.show(5, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|tokens                                                                                                                                                                                                                                    |id |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---+
|[I've, been, searching, for, the, right, words, to, thank, you, for, this, breather., I, promise, i, wont, take, your, help, for, granted, and, will, fulfil, my, promise., You, have, been, wonderful, and, a, blessing, at, all, times.]|0  |
|[Free, entry, in, 2, a, wkly, comp,

In [15]:
from pyspark.ml.feature import CountVectorizer

count_vectorizer = CountVectorizer(inputCol="tokens", outputCol="features").fit(df)
print("Vocab length", len(count_vectorizer.vocabulary))
count_vectorizer.vocabulary[:10]

Vocab length 15725


['to', 'you', 'I', 'a', 'the', 'and', 'in', 'is', 'i', 'u']

In [0]:
from pyspark.mllib.util import MLUtils

def count_vectors_from_df(df):
  count_vectors_df = count_vectorizer.transform(df)
  vectors_df = MLUtils.convertVectorColumnsFromML(count_vectors_df, "features") # needed to conver SparseVector to Vector
  row_rdd = vectors_df.select("features").rdd
  return row_rdd.map(lambda each: each.asDict()['features'])

In [19]:
count_vectors_rdd = count_vectors_from_df(df)
count_vectors_rdd.take(5)

[SparseVector(15725, {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 2.0, 8: 1.0, 10: 3.0, 11: 1.0, 15: 1.0, 17: 1.0, 25: 1.0, 28: 1.0, 37: 1.0, 48: 1.0, 51: 1.0, 85: 2.0, 109: 1.0, 186: 1.0, 285: 1.0, 352: 1.0, 386: 1.0, 785: 1.0, 818: 1.0, 1182: 1.0, 1547: 1.0, 2051: 1.0, 2475: 1.0, 3923: 1.0, 6332: 1.0, 7674: 1.0, 14955: 1.0, 15068: 1.0}),
 SparseVector(15725, {0: 3.0, 3: 1.0, 6: 1.0, 18: 1.0, 146: 1.0, 279: 1.0, 294: 1.0, 298: 1.0, 329: 1.0, 443: 2.0, 667: 1.0, 744: 1.0, 825: 1.0, 1098: 1.0, 1125: 1.0, 2383: 1.0, 2663: 1.0, 2709: 2.0, 2964: 1.0, 3107: 1.0, 3855: 1.0, 4337: 1.0, 4741: 1.0, 5097: 1.0}),
 SparseVector(15725, {0: 1.0, 2: 1.0, 80: 2.0, 96: 1.0, 97: 1.0, 151: 1.0, 218: 1.0, 435: 1.0, 749: 1.0, 1620: 1.0, 4812: 1.0, 5409: 1.0}),
 SparseVector(15725, {0: 1.0, 7: 1.0, 11: 1.0, 14: 1.0, 26: 1.0, 27: 1.0, 49: 2.0, 116: 1.0, 346: 1.0, 641: 1.0, 742: 1.0, 981: 1.0, 1159: 1.0, 11050: 1.0, 14852: 1.0}),
 SparseVector(15725, {2: 1.0, 118: 1.0, 622: 1.0, 775: 1.0, 1036: 1.0, 2037: 1.0,

In [20]:
from pyspark.mllib.feature import IDF

count_vectors_rdd.cache()
idf = IDF().fit(count_vectors_rdd)
tfidf = idf.transform(count_vectors_rdd)
tfidf.take(5)

[SparseVector(15725, {0: 1.2259, 1: 1.5187, 2: 1.5962, 3: 1.6186, 4: 1.7501, 5: 4.1371, 8: 2.2401, 10: 6.8129, 11: 2.3581, 15: 2.435, 17: 2.5363, 25: 2.7993, 28: 2.9083, 37: 3.0959, 48: 3.2735, 51: 3.2878, 85: 7.7263, 109: 4.0004, 186: 4.5649, 285: 4.8877, 352: 5.1913, 386: 5.2241, 785: 5.9863, 818: 5.9173, 1182: 6.3227, 1547: 6.5459, 2051: 6.8336, 2475: 7.0159, 3923: 7.5267, 6332: 7.5267, 7674: 7.9322, 14955: 7.9322, 15068: 7.9322}),
 SparseVector(15725, {0: 3.6778, 3: 1.6186, 6: 2.0392, 18: 2.7038, 146: 4.3213, 279: 4.8641, 294: 4.9365, 298: 5.0144, 329: 5.1288, 443: 11.1616, 667: 5.735, 744: 5.8527, 825: 5.9863, 1098: 6.2274, 1125: 6.2274, 2383: 7.0159, 2663: 7.0159, 2709: 15.0534, 2964: 7.239, 3107: 7.239, 3855: 7.239, 4337: 7.5267, 4741: 7.5267, 5097: 7.5267}),
 SparseVector(15725, {0: 1.2259, 2: 1.5962, 80: 8.0204, 96: 3.9068, 97: 4.0404, 151: 4.3078, 218: 4.7133, 435: 5.3672, 749: 5.8527, 1620: 6.5459, 4812: 7.5267, 5409: 7.5267}),
 SparseVector(15725, {0: 1.2259, 7: 2.1286, 11:

In [0]:
def labeled_point_per_sample(sample_and_vector):
  sample = sample_and_vector[0]
  feature_vector = sample_and_vector[1]
  label = 1 if sample.split("\t")[0] == 'spam' else 0
  return LabeledPoint(label, feature_vector)

In [22]:
labeled_points_rdd = corpus.zip(tfidf).map(labeled_point_per_sample)
print("size of data:", len(labeled_points_rdd.collect()))
labeled_points_rdd.take(5)

size of data: 5570


[LabeledPoint(0.0, (15725,[0,1,2,3,4,5,8,10,11,15,17,25,28,37,48,51,85,109,186,285,352,386,785,818,1182,1547,2051,2475,3923,6332,7674,14955,15068],[1.2259317666894618,1.518723712293513,1.5962422858711534,1.618634623183775,1.7500977627442384,4.137102987725546,2.24013545102309,6.812879427670396,2.358129301479453,2.435014444167668,2.5362849745739693,2.7993297426403654,2.908302148614594,3.0959007625093924,3.273471716544749,3.2877917703194974,7.726311830446119,4.000357036736545,4.564886839474396,4.887660231737447,5.191342645535669,5.22413246835866,5.986272520405557,5.917279648918606,6.322744757026769,6.545888308340979,6.833570380792761,7.015891937586715,7.526717561352706,7.526717561352706,7.93218266946087,7.93218266946087,7.93218266946087])),
 LabeledPoint(1.0, (15725,[0,3,6,18,146,279,294,298,329,443,667,744,825,1098,1125,2383,2663,2709,2964,3107,3855,4337,4741,5097],[3.6777953000683854,1.618634623183775,2.0391581951661406,2.703751430377,4.321264756816646,4.864129734327253,4.93645039590687

In [0]:
spam_model_tfidf = LogisticRegressionWithSGD.train(labeled_points_rdd)

### Evaluate

In [24]:
def feature_vector_from_text(text):
  test_df = spark.createDataFrame([(0, text.split(" "))], ["id", "tokens"])
  test_count_vectors_rdd = count_vectors_from_df(test_df)
  return idf.transform(test_count_vectors_rdd)

print(spam_model_tfidf.predict(feature_vector_from_text("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's")).collect())
print(spam_model_tfidf.predict(feature_vector_from_text("I've been searching for the right words to thank you for this breather")).collect())
print(spam_model_tfidf.predict(feature_vector_from_text("WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.")).collect())
print(spam_model_tfidf.predict(feature_vector_from_text("URGENT, IMPORTANT INFORMATION FOR O2 USER. TODAY IS YOUR LUCKY DAY! 2 FIND OUT WHY LOG ONTO HTTP://WWW.URAWINNER.COM THERE IS A FANTASTIC SURPRISE AWAITING FOR YOU")).collect())
print(spam_model_tfidf.predict(feature_vector_from_text("THING R GOOD THANX GOT EXAMS IN MARCH IVE DONE NO REVISION? IS FRAN STILL WITH BOYF? IVE GOTTA INTERVIW 4 EXETER BIT WORRIED!x")).collect())

[1]
[0]
[1]
[1]
[0]


**Observation:** Bingo! Looks like the model is working well